In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
cd drive/MyDrive/

/content/drive/MyDrive


In [4]:
train = pd.read_csv('traffic_V6.csv')
test = pd.read_csv('test_traffic_V6.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (249944, 108)
테스트 데이터 크기: (50000, 107)


In [5]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 104


In [6]:
cat_features = ["layout_type"]

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        verbose=100
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
0:	learn: 14.1959473	test: 14.2513678	best: 14.2513678 (0)	total: 593ms	remaining: 9m 52s
100:	learn: 9.1647430	test: 9.2522748	best: 9.2522748 (100)	total: 20.2s	remaining: 2m 59s
200:	learn: 8.9704644	test: 9.0876639	best: 9.0876639 (200)	total: 38.3s	remaining: 2m 32s
300:	learn: 8.7757993	test: 8.9240662	best: 8.9240662 (300)	total: 57.3s	remaining: 2m 12s
400:	learn: 8.6219697	test: 8.8030375	best: 8.8030375 (400)	total: 1m 16s	remaining: 1m 53s
500:	learn: 8.4814919	test: 8.6941311	best: 8.6941311 (500)	total: 1m 35s	remaining: 1m 34s
600:	learn: 8.3645241	test: 8.6066796	best: 8.6066796 (600)	total: 1m 53s	remaining: 1m 15s
700:	learn: 8.2571052	test: 8.5249305	best: 8.5249305 (700)	total: 2m 12s	remaining: 56.5s
800:	learn: 8.1601713	test: 8.4551012	best: 8.4551012 (800)	total: 2m 30s	remaining: 37.5s
900:	learn: 8.0678001	test: 8.3885921	best: 8.3885921 (900)	total: 2m 50s	remaining: 18.7s
999:	learn: 7.9818650	test: 8.3264818	best: 8.3264818 (999)	total: 3m 8s	re

In [8]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 8.3029


In [9]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V23.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
